# Vesuvius Challenge - Kaggle Training

This notebook is designed to run on Kaggle with GPU acceleration.
It loads data from Kaggle datasets and trains the ink detection model.

## Setup

Install dependencies and configure paths for Kaggle environment.

In [ ]:
# Install additional dependencies if needed
# !pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import sys
import gc
import numpy as np
from pathlib import Path

# Check if running on Kaggle
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    # Kaggle paths
    DATA_DIR = Path('/kaggle/input/vesuvius-challenge-ink-detection')
    OUTPUT_DIR = Path('/kaggle/working')
    # Add source directory to path
    sys.path.insert(0, '/kaggle/input/vesuvius-src/src')
else:
    # Local paths
    DATA_DIR = Path('../data/raw')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, '..')

print(f'Running on Kaggle: {IS_KAGGLE}')
print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

## Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Data
    'data_dir': str(DATA_DIR),
    'in_channels': 16,
    'size': 256,
    
    # Model
    'features': [32, 64, 128, 256],
    'out_channels': 1,
    'use_attention': False,
    
    # Training
    'epochs': 20,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 4,
    
    # Early stopping
    'patience': 5,
    
    # Seed
    'seed': 42,
}

print('Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

In [ ]:
from src.utils.helpers import set_seed

set_seed(CONFIG['seed'])

## Data Loading

In [ ]:
from src.data.loader import VesuviusDataset, create_dataloaders

train_loader, val_loader = create_dataloaders(
    CONFIG,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)

print(f'Train samples: {len(train_loader)}')
print(f'Val samples: {len(val_loader)}')

## Model

In [ ]:
from src.models.unet3d import create_model
from src.utils.helpers import get_device

device = get_device()
print(f'Using device: {device}')

model = create_model(CONFIG)
print(f'Model parameters: {model.get_params_count():,}')

## Training

In [ ]:
from src.utils.helpers import (
    AverageMeter,
    EarlyStopping,
    dice_coefficient,
    fbeta_score,
    save_checkpoint,
)

# Initialize training components
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=CONFIG['learning_rate'],
#     weight_decay=CONFIG['weight_decay'],
# )
# criterion = torch.nn.BCEWithLogitsLoss()
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=CONFIG['epochs']
# )

early_stopping = EarlyStopping(patience=CONFIG['patience'], mode='max')
best_score = 0.0

In [ ]:
# Training loop
for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    
    # Training phase
    train_loss = AverageMeter()
    
    # model.train()
    # for batch_idx, (inputs, labels) in enumerate(train_loader):
    #     inputs, labels = inputs.to(device), labels.to(device)
    #     
    #     optimizer.zero_grad()
    #     outputs = model(inputs)
    #     loss = criterion(outputs, labels)
    #     loss.backward()
    #     optimizer.step()
    #     
    #     train_loss.update(loss.item())
    
    # Validation phase
    val_loss = AverageMeter()
    val_dice = AverageMeter()
    val_fbeta = AverageMeter()
    
    # model.eval()
    # with torch.no_grad():
    #     for inputs, labels in val_loader:
    #         inputs, labels = inputs.to(device), labels.to(device)
    #         outputs = model(inputs)
    #         loss = criterion(outputs, labels)
    #         
    #         preds = torch.sigmoid(outputs).cpu().numpy()
    #         targets = labels.cpu().numpy()
    #         
    #         val_loss.update(loss.item())
    #         val_dice.update(dice_coefficient(preds, targets))
    #         val_fbeta.update(fbeta_score(preds, targets))
    
    print(f'Train Loss: {train_loss.avg:.4f}')
    print(f'Val Loss: {val_loss.avg:.4f}, Dice: {val_dice.avg:.4f}, F0.5: {val_fbeta.avg:.4f}')
    
    # Save best model
    if val_fbeta.avg > best_score:
        best_score = val_fbeta.avg
        save_checkpoint(
            model, None, epoch, val_loss.avg,
            str(OUTPUT_DIR / 'best_model.pth'),
        )
        print(f'New best model! F0.5: {best_score:.4f}')
    
    # Early stopping
    if early_stopping(val_fbeta.avg):
        print(f'Early stopping at epoch {epoch + 1}')
        break
    
    # Cleanup
    gc.collect()

print(f'\nTraining complete! Best F0.5: {best_score:.4f}')

## Inference (for submission)

In [ ]:
# Load best model for inference
# checkpoint = torch.load(OUTPUT_DIR / 'best_model.pth')
# model.load_state_dict(checkpoint['model_state'])
# model.eval()

# Generate predictions for test set
# predictions = []
# with torch.no_grad():
#     for inputs in test_loader:
#         inputs = inputs.to(device)
#         outputs = model(inputs)
#         preds = torch.sigmoid(outputs).cpu().numpy()
#         predictions.append(preds)

print('Inference complete!')

## Create Submission

In [ ]:
# Create submission file
# submission = create_submission(predictions, fragment_ids)
# submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

print('Submission file created!')